# Lesson 17 Lab — Fused LayerNorm and RMSNorm

**Puzzle:** When row statistics, FP32 accumulation, and affine output change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates row statistics, FP32 accumulation, and affine output and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

RMSNorm reads a row, accumulates squared values in FP32, computes reciprocal RMS, applies a learned weight, and writes once. LayerNorm adds mean subtraction. Both are memory-sensitive frequent operators whose hidden width sets a practical one-program limit.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["row statistics, FP32 accumulation, and affine output"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

A forward-only result says nothing about backward input and weight gradients, which can require additional reductions and synchronization.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 17
LESSON_TITLE = 'Fused LayerNorm and RMSNorm'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260830
}


## 5. Freeze the experiment

**Experiment:** Implement fused RMSNorm forward and compare it with an equivalent eager PyTorch expression.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.017280000261962414,
  "secondary": 0.04447999969124794,
  "max_abs_error": 1.9073486328125e-06,
  "passed": true,
  "details": {
    "triton_samples_ms": [
      0.028991999104619026,
      0.020959999412298203,
      0.018751999363303185,
      0.017632000148296356,
      0.01744000054895878,
      0.019007999449968338,
      0.016704000532627106,
      0.016863999888300896,
      0.01587199978530407,
      0.017472000792622566,
      0.016063999384641647,
      0.01635199971497059,
      0.015776000916957855,
      0.01865600049495697,
      0.015968000516295433,
      0.017055999487638474,
      0.01756799966096878,
      0.01772800087928772,
      0.01711999997496605,
      0.015584000386297703
    ],
    "pytorch_eager_samples_ms": [
      0.04771199822425842,
      0.04553600028157234,
      0.04460800066590309,
      0.04521600157022476,
      0.04428799822926521,
      0.05734400078654289,
      0.04572800174355507,
      0.045024000108242035,
      0.044863998

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Triton median | 0.0173 ms |
| PyTorch eager median | 0.0445 ms |
| Maximum absolute error | 1.907e-06 |
| Acceptance gate | true |


## 8. Explain without overclaiming

The fused RMSNorm forward path took 0.0173 ms versus 0.0445 ms for its eager expression, with max error 1.91e-06. Backward is outside this lab.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Adopt the forward kernel only for validated widths; treat training backward as a separate deliverable.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 17,
  "title": "Fused LayerNorm and RMSNorm",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260830
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.017280000261962414,
    "secondary": 0.04447999969124794,
    "max_abs_error": 1.9073486328125e-06,
    "passed": true,
    "details": {
      "triton_samples_ms": [
        0.028991999104619026,
        0.020959999412298203,
        0.018751999363303185,
        0.017632000148296356,
        0.01744000054895878,
        0.019007999449968338,
        0.016704000532627106,
        0.016863999888300896,
        0.01587199978530407,
        0.017472000792622566,
        0.016063999384641647,
        0.01635199971497059,
        0.015776000916957855,
        0.018656000494

## 10. Make the bounded decision

> Adopt the forward kernel only for validated widths; treat training backward as a separate deliverable.

**Failure analysis:** A forward-only result says nothing about backward input and weight gradients, which can require additional reductions and synchronization.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
